In [ ]:
import os
import sys
import pandas as pd
import xarray as xr

sys.path.append("./resources/library")

from tropical_cyclone.cyclone import LocClsModelInference

# ------------------------------------------------------------
# 1. Base path to the models shipped in the repository
# ------------------------------------------------------------
MODELS_DIR = "./data/models"

north_loc = os.path.join(MODELS_DIR, "bytestorm_loc_NHS")
north_cls = os.path.join(MODELS_DIR, "bytestorm_cls_NHS")

south_loc = os.path.join(MODELS_DIR, "bytestorm_loc_SHS")
south_cls = os.path.join(MODELS_DIR, "bytestorm_cls_SHS")

# ------------------------------------------------------------
# 2. Load a sample dataset
#    Assumes the file contains variables like:
#    - time
#    - lat
#    - lon
#    - vo_850
#    - msl
# ------------------------------------------------------------
# Example:
# ds = xr.open_mfdataset("./data/datasets/global/*.nc")
# For a quick example, replace with your own path:
ds = xr.open_mfdataset("./data/datasets/global/*.nc")

# ------------------------------------------------------------
# 3. Split by hemisphere
#    The project expects north/south lat slices
# ------------------------------------------------------------
ds_north = ds.sel(lat=slice(70, 0))
ds_south = ds.sel(lat=slice(0, -70))

# ------------------------------------------------------------
# 4. Instantiate the two model pairs
# ------------------------------------------------------------
infer_north = LocClsModelInference(
    model_loc_dir=north_loc,
    model_cls_dir=north_cls,
    device="cpu"  # or "cuda" if available
)

infer_south = LocClsModelInference(
    model_loc_dir=south_loc,
    model_cls_dir=south_cls,
    device="cpu"
)

# ------------------------------------------------------------
# 5. Run inference
#    threshold is the classification threshold for cyclone detection
# ------------------------------------------------------------
detections_north = infer_north.predict(
    ds=ds_north,
    patch_size=40,
    threshold=0.5,
    batch_size=1,
    num_workers=1
)

detections_south = infer_south.predict(
    ds=ds_south,
    patch_size=40,
    threshold=0.5,
    batch_size=1,
    num_workers=1
)

# ------------------------------------------------------------
# 6. Merge the results
# ------------------------------------------------------------
all_detections = pd.concat(
    [detections_north, detections_south],
    ignore_index=True
)

print("North detections:", len(detections_north))
print("South detections:", len(detections_south))
print("Total detections:", len(all_detections))

print(all_detections.head())